# MLSM Lab 1 — Entrenamiento interactivo

El notebook se encarga de la interacción y la presentación. La lógica está en `run_lab1.py` y las definiciones de modelos en `model_definitions.py`. Ejecuta las celdas en orden.

In [ ]:
from run_lab1 import (
    load_dataset, select_models, evaluate_models,
    get_confusion_data, create_cv, display_results, plot_confusion_matrix,
)


## 1. Seleccionar dataset

In [ ]:
print("1 -> GenreAll | 2 -> features-30-sec | 3 -> features-3-sec")
dataset_choice = input("Dataset [1/2/3]: ").strip()

## 2. Cargar datos

In [ ]:
dataset_name, filename, X, y, class_counts = load_dataset(dataset_choice)
print(f"{dataset_name}: {len(y)} muestras, {X.shape[1]} características")
display_results(class_counts.to_frame())

## 3. Seleccionar modelo

In [ ]:
print("l -> Logistic Regression | k -> k-NN | d -> Decision Tree | a -> ANN | all -> Todos")
model_choice = input("Modelo [l/k/d/a/all]: ")

In [ ]:
selected_models = select_models(dataset_name, X.shape[1], model_choice)
print("Modelos:", ", ".join(selected_models))

cv = create_cv()
results_df = evaluate_models(X, y, selected_models, cv)
print(f"Configuraciones evaluadas: {len(results_df)}")

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results_df = evaluate_models(X, y, selected_models, cv)
print(f"Configuraciones evaluadas: {len(results_df)}")

display_results(results_df)


In [ ]:
display(results_df.style.format({"Train accuracy": "{:.4f}", "CV accuracy": "{:.4f}", "Overfitting gap": "{:.4f}"}))

## 6. Seleccionar fila para la matriz de confusión

La matriz usa predicciones *out-of-fold* de validación cruzada.

In [ ]:
row_choice = input(f"Índice de fila [0–{len(results_df)-1}] o q para omitir: ").strip().lower()

## 7. Matriz de confusión

In [ ]:
if row_choice == "q":
    print("Matriz omitida.")
else:
    row_index = int(row_choice)
    matrix, classes, row = get_confusion_data(X, y, selected_models, results_df, row_index, cv)
    print(f"{row['Algorithm']} | CV accuracy: {row['CV accuracy']:.4f} | Overfitting gap: {row['Overfitting gap']:.4f}\nParámetros: {row['Parameters']}")
    plot_confusion_matrix(matrix, classes)